# 09. 실험 관리, 평가표, 모델 카드

연구를 논문으로 만들려면 `무엇을 바꿨고`, `왜 좋아졌고`, `어떤 조건에서 실패했는지`가 남아야 합니다.
이 노트북은 CSV 기반의 경량 실험 추적과 모델 카드 생성을 연습합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
experiments = pd.DataFrame([
    ("E001", "text_only", "OCR text only", 0.62, 0.00, 120, "baseline"),
    ("E002", "ocr_layout", "OCR + layout region", 0.71, 0.00, 145, "layout helps table fields"),
    ("E003", "asr_ocr", "ASR command + OCR", 0.74, 0.08, 180, "speech intent reduces ambiguity"),
    ("E004", "asr_ocr_sound_gate", "ASR + OCR + sound safety gate", 0.76, 0.03, 210, "fewer unsafe actions"),
], columns=["run_id", "system", "description", "qa_accuracy", "unsafe_action_rate", "latency_ms", "note"])

experiments["score"] = experiments["qa_accuracy"] - 2.0 * experiments["unsafe_action_rate"] - 0.0002 * experiments["latency_ms"]
experiments.to_csv(ARTIFACTS / "experiment_log.csv", index=False, encoding="utf-8-sig")
experiments.sort_values("score", ascending=False)


In [ ]:
best = experiments.sort_values("score", ascending=False).iloc[0].to_dict()
model_card = f'''
# Model Card: {best["system"]}

## Intended Use
현장형 문서/상황 QA에서 음성 명령, OCR 텍스트, 환경음 게이트를 결합해 답변 또는 안전 확인을 수행한다.

## Metrics
- QA accuracy: {best["qa_accuracy"]:.3f}
- Unsafe action rate: {best["unsafe_action_rate"]:.3f}
- Latency: {best["latency_ms"]:.0f} ms

## Data Handling
회사명, 고객명, 소속명은 모든 보고 산출물에서 xxx로 마스킹한다.

## Limitations
낮은 대비 문서, 겹쳐 말하기, 다국어 숫자 코드에서 오류가 발생할 수 있다.
'''
path = ARTIFACTS / "MODEL_CARD.md"
path.write_text(dedent(model_card).strip() + "\n", encoding="utf-8")
print(path)


## 리더십 연결

팀 리드 역할에서는 실험표가 의사결정 도구입니다.
코드 리뷰 때 `정확도만 올랐는가`, `latency/safety/regression은 어떤가`, `재현 가능한가`를 함께 봅니다.
